# Denoise images with FastAPI

Build an image request around a confirmed graph, compare two conditional routes, and inspect an invalid request. You should know basic Python and HTTP request/response concepts; FastAPI familiarity helps. Pillow performs the image operation, while Jayrun manages execution and the reusable HTTP client.

## Before you begin

Run the cells in order in a fresh kernel. Keep the complete repository and bundled tutorial data together. The setup below locates your checkout; in Colab it accepts a complete source ZIP. The notebook imports the adjacent Python implementation, so operators and resources have one canonical source.


In [ ]:
from pathlib import Path
import importlib.util
import io
import subprocess
import sys
import zipfile

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "jayrun").is_dir() and (p / "tutorials").is_dir()), None)
if ROOT is None and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()  # Upload the complete Jayrun project ZIP, not just this notebook.
    if len(uploaded) != 1:
        raise ValueError("upload exactly one project ZIP")
    destination = Path("/content/jayrun_tutorial_source")
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
        if sum(item.file_size for item in archive.infolist()) > 500_000_000:
            raise ValueError("archive exceeds this notebook's extraction limit")
        for item in archive.infolist():
            path = (destination / item.filename).resolve()
            if not path.is_relative_to(destination.resolve()) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("unsafe archive path or symbolic link")
        archive.extractall(destination)
    candidates = [destination, *destination.iterdir()]
    ROOT = next((p for p in candidates if (p / "jayrun").is_dir() and (p / "tutorials").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run Jupyter from the extracted project, or upload the full project ZIP in Colab.")
sys.path.insert(0, str(ROOT))
print("Tutorial source:", ROOT)


## Prepare the environment

The next cell checks the packages this lesson uses. Locally, install missing tutorial packages before continuing. In Colab, it installs only missing non-PyTorch packages. For PyTorch lessons, provide a suitable existing PyTorch build; the cell does not replace it.


In [ ]:
required = {'packaging': 'packaging>=25', 'fastapi': 'fastapi', 'httpx': 'httpx', 'PIL': 'Pillow'}
missing = [requirement for name, requirement in required.items() if importlib.util.find_spec(name) is None]
# Colab provides PyTorch. Never replace an existing CPU/CUDA installation here.
if "torch" in missing:
    raise RuntimeError("Install the PyTorch build appropriate for your hardware before running this lesson.")
if missing:
    if not IN_COLAB:
        raise RuntimeError(f"Install these optional tutorial dependencies first: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
from packaging.version import Version
import packaging
if Version(packaging.__version__) < Version("25"):
    raise RuntimeError("This source archive requires packaging >= 25; update it explicitly before proceeding.")


## 1. Trace the route

**ImageSource → LoadImage → RouteImage → EncodeImage → PNG response**. RouteImage has two distinct output artifacts. One carries image bytes and the other receives None, which skips its consumer. The graph declaration stays the same for both requests. The request's ConfigContext chooses the route.


In [ ]:
import asyncio
import inspect
import io
from PIL import Image
from IPython.display import display
from fastapi.testclient import TestClient
from tutorials import denoise_images as lesson

print(inspect.getsource(lesson.build_graph))


## 2. Make the difference visible

Create a white image with a black pixel. The median filter should remove that isolated pixel; the preserving route should keep it. We send raw PNG bytes, not multipart form data. The same requests work against Uvicorn at /images/upload.


In [ ]:
buffer = io.BytesIO()
with Image.new("RGB", (16, 16), "white") as image:
    image.putpixel((8, 8), (0, 0, 0))
    image.save(buffer, format="PNG")
source_png = buffer.getvalue()
with Image.open(io.BytesIO(source_png)) as image:
    display(image.resize((160, 160), Image.Resampling.NEAREST))


## 3. Exercise the application lifespan

TestClient's context manager starts and closes the app lifespan, including the Engine. Calls run in a worker thread so they do not nest a synchronous service demonstration inside Jupyter's loop. Each request gets its own run; both responses should be HTTP 200 with PNG content.


In [ ]:
def compare_routes():
    with TestClient(lesson.create_app()) as client:
        clean = client.post("/images/upload?denoise=true", content=source_png)
        preserved = client.post("/images/upload?denoise=false", content=source_png)
    assert clean.status_code == preserved.status_code == 200
    assert clean.headers["content-type"] == "image/png"
    return clean.content, preserved.content

clean_png, preserved_png = await asyncio.to_thread(compare_routes)
for label, content in (("Preserved", preserved_png), ("Denoised", clean_png)):
    with Image.open(io.BytesIO(content)) as image:
        print(label, "center pixel:", image.getpixel((8, 8)))
        display(image.resize((160, 160), Image.Resampling.NEAREST))
with Image.open(io.BytesIO(clean_png)) as image:
    assert image.getpixel((8, 8)) == (255, 255, 255)
with Image.open(io.BytesIO(preserved_png)) as image:
    assert image.getpixel((8, 8)) == (0, 0, 0)


## 4. Inspect a controlled failure

An invalid image should return 400. A URL is refused with 403 unless its host is explicitly allowed. These are application policies. A wait timeout is handled differently: the application requests abort, drains the run, then returns 504; waiting alone does not cancel accepted work.


In [ ]:
def check_invalid_requests():
    with TestClient(lesson.create_app()) as client:
        invalid = client.post("/images/upload", content=b"not an image")
        disabled = client.post("/images/url", json={"url": "http://127.0.0.1/image.png"})
        return invalid.status_code, disabled.status_code

statuses = await asyncio.to_thread(check_invalid_requests)
print("Invalid image / disabled URL:", statuses)
assert statuses == (400, 403)


## Try it yourself

Move the black pixel in the preparation cell and update the inspected coordinates and pixel assertions before rerunning the comparison. You can also add a larger black patch and inspect it separately. A median filter removes isolated noise; it need not remove a larger region. Keep both HTTP status and decoded pixels in your inspection.

Serve with one ASGI worker. Public exposure needs application authentication and upload limits. The host allowlist is not complete SSRF/DNS-rebinding protection; redirects are not followed. The HTTP resource is used for URL loading, while these offline requests exercise the byte-upload path.
